# Tugas Mandiri Data Wrangling: Pertemuan 2
**Nama Mahasiswa:** Zahrotul Afta Ramadhana  
**NIM:** 25200017  
**Kasus:** Analisis Performa Kampanye Kupon Diskon Akhir Bulan, PT Nusantara Retail Mandiri

## Bagian A: Analisis Konseptual & Tata Kelola Data

### A.1 Penerjemahan Kebutuhan Bisnis menjadi Kebutuhan Teknis Data

Kebutuhan dari Head of Business Strategy terdiri dari tiga pertanyaan:
(a) berapa total transaksi yang menggunakan kupon, (b) bagaimana profil loyalitas pelanggan yang memakainya, dan (c) apakah pesanan tersebut sudah terkirim.

| Kebutuhan Bisnis | Atribut | Tipe Data | Sumber Asal |
|---|---|---|---|
| Total transaksi berkupon | `order_id` (primary key) | string | CSV POS |
| | `order_date` | datetime | CSV POS |
| | `total_amount` | integer/float | CSV POS |
| | `coupon_code` (kosong = tanpa kupon) | string | CSV POS |
| Profil loyalitas pelanggan | `customer_id` (kunci join) | string | CSV POS + SQL CRM |
| | `customer_name` | string | SQL CRM |
| | `membership_tier` | string (kategori) | SQL CRM |
| | `city` | string | SQL CRM |
| | `is_active` (filter pelanggan aktif) | integer (0/1) | SQL CRM |
| Status pengiriman pesanan | `order_id` (kunci join) | string | REST API Logistik |
| | `tracking_number` | string | REST API Logistik |
| | `delivery_status` | string (kategori) | REST API Logistik |
| | `courier.name` → `courier_name` | string | REST API Logistik (nested JSON) |
| | `courier.service` → `courier_service` | string | REST API Logistik (nested JSON) |

### A.2 Matriks Peran RACI

Keterangan: **R** = Responsible (mengerjakan), **A** = Accountable (penanggung jawab akhir, hanya satu per aktivitas), **C** = Consulted (dimintai masukan), **I** = Informed (diberi tahu).

| Aktivitas | Head of Business Strategy (Data Owner) | Data Management Office / Data Steward | Tim Data Wrangling (Data Custodian/Engineer) | BI Analyst (Data User) |
|---|---|---|---|---|
| Menetapkan definisi "transaksi berkupon" dan KPI kampanye | A/R | C | C | I |
| Menetapkan standar kualitas data dan kamus data | A | R | C | C |
| Membangun pipeline ingestion dan integrasi data | I | C | A/R | I |
| Validasi kualitas data (automated assertion) | I | A | R | I |
| Kebijakan akses data pelanggan (PII) | A | R | C | I |
| Analisis dan pelaporan performa kampanye | A | I | C | R |

## Bagian B: Python & Data Ingestion Pipeline

In [27]:
import pandas as pd, numpy as np, sqlite3, json

rng = np.random.default_rng(42)
n = 200
orders = [f"ORD{1000+i}" for i in range(n)]

# 1) CSV POS
df = pd.DataFrame({
    "order_id": orders,
    "customer_id": [f"C{rng.integers(1, 41):03d}" for _ in range(n)],
    "order_date": pd.Timestamp("2026-09-01") + pd.to_timedelta(rng.integers(0, 30, n), unit="D"),
    "total_amount": rng.integers(50_000, 1_500_000, n),
    "coupon_code": rng.choice(["AKHIRBULAN20", "DISKON10", None], n, p=[.4, .2, .4]),
})
df.to_csv("transaksi_pos.csv", index=False)

# 2) SQLite CRM
crm = pd.DataFrame({
    "customer_id": [f"C{i:03d}" for i in range(1, 41)],
    "customer_name": [f"Pelanggan {i}" for i in range(1, 41)],
    "membership_tier": rng.choice(["Silver", "Gold", "Platinum"], 40),
    "city": rng.choice(["Semarang", "Jakarta", "Surabaya", "Bandung"], 40),
    "is_active": rng.choice([0, 1], 40, p=[.15, .85]),
})
con = sqlite3.connect("crm_loyalty.db")
crm.to_sql("customers", con, if_exists="replace", index=False)
con.close()

# 3) Mock JSON API (bersarang / nested), sengaja 30 order belum ada datanya
results = [{
    "order_id": o,
    "tracking_number": f"TRK{rng.integers(10**8, 10**9)}",
    "delivery_status": str(rng.choice(["Delivered", "In Transit", "Pending"])),
    "courier": {"name": str(rng.choice(["JNE", "J&T", "SiCepat"])), "service": "REG"},
} for o in orders[:170]]
with open("logistik_mock.json", "w") as f:
    json.dump({"results": results}, f)

In [28]:
# TUGAS MANDIRI DATA WRANGLING - PERTEMUAN 2
# Nama Mahasiswa : Zahrotul Afta Ramadhana
# NIM            : 2520017

import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine

In [29]:
df_transaksi = pd.read_csv(
    "transaksi_pos.csv",
    dtype={"order_id": str, "customer_id": str},
    parse_dates=["order_date"],
)
print(df_transaksi.dtypes)
print(df_transaksi.shape)
df_transaksi.head()

order_id                object
customer_id             object
order_date      datetime64[ns]
total_amount             int64
coupon_code             object
dtype: object
(200, 5)


,order_id,customer_id,order_date,total_amount,coupon_code
0,ORD1000,C004,2026-09-11,1497140,NaN
1,ORD1001,C031,2026-09-28,1177237,AKHIRBULAN20
2,ORD1002,C027,2026-09-15,514604,DISKON10
3,ORD1003,C018,2026-09-21,1459148,DISKON10
4,ORD1004,C018,2026-09-14,773995,NaN


In [30]:
engine = create_engine("sqlite:///crm_loyalty.db")

query_crm = """
    SELECT customer_id, customer_name, membership_tier, city, is_active
    FROM customers
    WHERE is_active = 1
"""
df_crm = pd.read_sql(query_crm, con=engine)
df_crm["customer_id"] = df_crm["customer_id"].astype(str)

print(df_crm.shape)
df_crm.head()

(35, 5)


,customer_id,customer_name,membership_tier,city,is_active
0,C001,Pelanggan 1,Platinum,Surabaya,1
1,C002,Pelanggan 2,Platinum,Bandung,1
2,C003,Pelanggan 3,Gold,Jakarta,1
3,C004,Pelanggan 4,Platinum,Jakarta,1
4,C006,Pelanggan 6,Platinum,Semarang,1


In [31]:
api_url = "https://raw.githubusercontent.com/AFTA3009/Tugas1-Data-Wrangling/refs/heads/main/logistik_mock.json"

response = requests.get(api_url, timeout=10)
response.raise_for_status()
data_logistik = response.json()

df_logistik = pd.json_normalize(data_logistik["results"])
df_logistik = df_logistik.rename(columns={
    "courier.name": "courier_name",
    "courier.service": "courier_service",
})
df_logistik["order_id"] = df_logistik["order_id"].astype(str)

print(df_logistik.columns.tolist())
df_logistik.head()

['order_id', 'tracking_number', 'delivery_status', 'courier_name', 'courier_service']


,order_id,tracking_number,delivery_status,courier_name,courier_service
0,ORD1000,TRK624625493,In Transit,SiCepat,REG
1,ORD1001,TRK578269936,In Transit,JNE,REG
2,ORD1002,TRK119744331,In Transit,SiCepat,REG
3,ORD1003,TRK667749016,In Transit,JNE,REG
4,ORD1004,TRK107692057,In Transit,JNE,REG


### B.4 Integrasi Data Multi-Sumber (Merging)

**Alasan pemilihan jenis join:**

**1. Transaksi POS ⟶ CRM: `LEFT JOIN`**
Tabel transaksi adalah "fakta utama" yang ingin dihitung. Data CRM hanya berisi pelanggan aktif (`is_active = 1`). Jika memakai `INNER JOIN`, transaksi milik pelanggan non-aktif atau yang tidak terdaftar akan hilang dari hasil, sehingga total transaksi berkupon menjadi lebih kecil dari kenyataan (*undercount*). Dengan `LEFT JOIN`, seluruh transaksi tetap terhitung, sedangkan atribut loyalitas bernilai `NaN` untuk pelanggan non-aktif dan dapat dianalisis sebagai kelompok tersendiri.

**2. Hasil gabungan ⟶ Logistik API: `LEFT JOIN`**
Pesanan yang belum diserahkan ke ekspedisi belum memiliki data pengiriman. Pesanan tersebut tetap harus muncul, dan nilai `NaN` pada status pengirimannya justru menjawab pertanyaan bisnis, yaitu bahwa pesanan itu belum memiliki data pengiriman.

**3. Pengaman integritas data:**
Parameter `validate="m:1"` (banyak transaksi per pelanggan) dan `validate="1:1"` (satu pesanan satu data kirim) memastikan tidak terjadi duplikasi baris akibat kunci join yang tidak unik. Selain itu, `assert` membandingkan jumlah baris akhir dengan jumlah transaksi awal.

In [32]:
n_awal = len(df_transaksi)

# POS -> CRM: LEFT JOIN
df_master = df_transaksi.merge(
    df_crm, on="customer_id", how="left", validate="m:1"
)

# hasil -> Logistik: LEFT JOIN
df_master_analisis = df_master.merge(
    df_logistik, on="order_id", how="left", validate="1:1"
)

assert len(df_master_analisis) == n_awal, "Jumlah baris berubah setelah merge!"

print(df_master_analisis.shape)
df_master_analisis.head()

(200, 13)


,order_id,customer_id,order_date,total_amount,coupon_code,customer_name,membership_tier,city,is_active,tracking_number,delivery_status,courier_name,courier_service
0,ORD1000,C004,2026-09-11,1497140,NaN,Pelanggan 4,Platinum,Jakarta,1.0,TRK624625493,In Transit,SiCepat,REG
1,ORD1001,C031,2026-09-28,1177237,AKHIRBULAN20,Pelanggan 31,Gold,Surabaya,1.0,TRK578269936,In Transit,JNE,REG
2,ORD1002,C027,2026-09-15,514604,DISKON10,Pelanggan 27,Silver,Bandung,1.0,TRK119744331,In Transit,SiCepat,REG
3,ORD1003,C018,2026-09-21,1459148,DISKON10,Pelanggan 18,Gold,Bandung,1.0,TRK667749016,In Transit,JNE,REG
4,ORD1004,C018,2026-09-14,773995,NaN,Pelanggan 18,Gold,Bandung,1.0,TRK107692057,In Transit,JNE,REG


In [33]:
def generate_data_dictionary(df, n_sample=3):
    rows = []
    for col in df.columns:
        n_missing = df[col].isnull().sum()
        sample = df[col].dropna().unique()[:n_sample]
        rows.append({
            "Nama Kolom": col,
            "Tipe Data (dtype)": str(df[col].dtype),
            "Jumlah Missing": int(n_missing),
            "Missing (%)": round(n_missing / len(df) * 100, 2),
            "Jumlah Nilai Unik": int(df[col].nunique()),
            "Sampel Nilai": ", ".join(map(str, sample)) if len(sample) else "-",
        })
    return pd.DataFrame(rows)

kamus_df = generate_data_dictionary(df_master_analisis)
kamus_df

,Nama Kolom,Tipe Data (dtype),Jumlah Missing,Missing (%),Jumlah Nilai Unik,Sampel Nilai
0,order_id,object,0,0.0,200,"ORD1000, ORD1001, ORD1002"
1,customer_id,object,0,0.0,38,"C004, C031, C027"
2,order_date,datetime64[ns],0,0.0,30,"2026-09-11 00:00:00, 2026-09-28 00:00:00, 2026..."
3,total_amount,int64,0,0.0,200,"1497140, 1177237, 514604"
4,coupon_code,object,85,42.5,2,"AKHIRBULAN20, DISKON10"
5,customer_name,object,26,13.0,33,"Pelanggan 4, Pelanggan 31, Pelanggan 27"
6,membership_tier,object,26,13.0,3,"Platinum, Gold, Silver"
7,city,object,26,13.0,4,"Jakarta, Surabaya, Bandung"
8,is_active,float64,26,13.0,1,1.0
9,tracking_number,object,30,15.0,170,"TRK624625493, TRK578269936, TRK119744331"


In [34]:
kamus_df.to_csv("kamus_data_hasil.csv", index=False)
print("[INFO] Kamus data disimpan ke kamus_data_hasil.csv")

[INFO] Kamus data disimpan ke kamus_data_hasil.csv


In [35]:
# 1) Primary key tidak boleh duplikat
assert df_master_analisis["order_id"].is_unique, \
    "Fatal: Terdeteksi order_id duplikat!"

# 2) Nilai transaksi tidak boleh negatif
assert (df_master_analisis["total_amount"] >= 0).all(), \
    "Fatal: Ada nilai total_amount negatif!"

# 3) Tanggal pesanan tidak boleh melebihi hari ini
today = pd.Timestamp.today().normalize()
assert (df_master_analisis["order_date"] <= today).all(), \
    "Fatal: Ada order_date di masa depan!"

print("[SUCCESS] Seluruh validasi data lulus!")

[SUCCESS] Seluruh validasi data lulus!


In [36]:
df_uji = df_master_analisis.copy()
df_uji.loc[0, "total_amount"] = -1000

try:
    assert (df_uji["total_amount"] >= 0).all(), "Terdeteksi nilai negatif"
except AssertionError as e:
    print("[OK] Assertion berhasil mendeteksi data buruk:", e)

[OK] Assertion berhasil mendeteksi data buruk: Terdeteksi nilai negatif


In [37]:
df_kupon = df_master_analisis[df_master_analisis["coupon_code"].notna()]

print("Total transaksi berkupon :", len(df_kupon))
print("Total nilai transaksi    :", df_kupon["total_amount"].sum())

print("\nProfil loyalitas:")
print(df_kupon["membership_tier"].fillna("Non-aktif / tidak terdaftar").value_counts())

print("\nStatus pengiriman:")
print(df_kupon["delivery_status"].fillna("Belum ada data").value_counts())

Total transaksi berkupon : 115
Total nilai transaksi    : 93330016

Profil loyalitas:
membership_tier
Silver                         37
Platinum                       37
Gold                           23
Non-aktif / tidak terdaftar    18
Name: count, dtype: int64

Status pengiriman:
delivery_status
Pending           35
In Transit        30
Delivered         27
Belum ada data    23
Name: count, dtype: int64
